# Day 09 — Pipeline kiểm nhãn tự động: Bước 1 → 5 trên Colab
**Nguyễn Duy Khánh — 2A202603055**

Chạy lại toàn bộ bài từ repo: **Bước 1** chuẩn bị, nhìn dữ liệu · **Bước 2** test 4 module · **Bước 3** calibrate 2 ngưỡng · **Bước 4** chạy pipeline (+ train raw vs clean trên GPU) · **Bước 5** tổng hợp và kiểm bài nộp.

Runtime → Change runtime type → **T4 GPU** (chỉ phần train cần GPU), rồi **Runtime → Run all**.

In [ ]:
# 0. Cấu hình
REPO_URL = "https://github.com/henrykhanh-ai/K5-L23-Day09-2A202603055-NGUYENDUYKHANH-AutoQC-Pipeline-Lab.git"
SEEDS = [0, 1]            # 2 seed để đo độ nhiễu
EPOCHS = 20
VARIANTS = ["raw", "clean"]
V_ARG = " ".join(VARIANTS); S_ARG = " ".join(map(str, SEEDS))
DO_TRAIN = True           # False nếu không có GPU: chỉ chạy Bước 2, 3, 4 (pipeline), 5 (check-core)

In [ ]:
# 1. Lấy code + cài môi trường
!nvidia-smi -L || echo "Không có GPU: phần train sẽ bỏ qua"
import os, subprocess, sys
repo = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not os.path.exists(repo):
    subprocess.run(["git", "clone", REPO_URL], check=True)
%cd {repo}
os.environ["PYTHONUTF8"] = "1"
!git log --oneline | head -8
!pip install -q -r requirements.txt
!pip install -q "ultralytics==8.4.214" matplotlib
import torch; HAS_GPU = torch.cuda.is_available(); print("GPU:", HAS_GPU)

## Bước 1 — Chuẩn bị, nhìn dữ liệu
Đề gốc: `pytest` ra **55 failed, 24 passed** khi 4 module còn là stub. Repo này đã viết xong nên sẽ ra **79 passed**; ô 1b tạm đưa 4 module về bản stub gốc để bạn thấy đúng trạng thái ban đầu, rồi khôi phục.

In [ ]:
# 1a. Môi trường + pytest (trạng thái hiện tại)
!python --version
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# 1b. Trạng thái ban đầu của đề: lấy 4 stub từ commit gốc f539663 -> pytest -> khôi phục bài làm
STUBS = "pipeline/checks/label_format.py pipeline/checks/second_opinion.py pipeline/steps/autofix.py pipeline/steps/gate.py"
!git fetch -q --unshallow 2>/dev/null; git checkout -q f539663 -- {STUBS}
!python -m pytest -q 2>&1 | tail -1
!git checkout -q HEAD -- {STUBS} && python -m pytest -q 2>&1 | tail -1

In [ ]:
# 1c. Bộ dữ liệu: 4 tập ảnh, 4 lớp
import pathlib, yaml
for d in ["raw", "calib", "gold_val", "test"]:
    ims = [p for p in pathlib.Path(f"data/{d}/images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    labs = list(pathlib.Path(f"data/{d}/labels").glob("*.txt")) if pathlib.Path(f"data/{d}/labels").exists() else []
    print(f"{d:9} ảnh={len(ims):4}  file nhãn={len(labs):4}")
print("data/raw/data.yaml:", yaml.safe_load(open("data/raw/data.yaml"))["names"], "| dự án:", yaml.safe_load(open("configs/pipeline.yaml"))["classes"])
print("đuôi ảnh raw:", sorted({p.suffix for p in pathlib.Path("data/raw/images").iterdir()}))
print("ảnh raw không có file nhãn:", sorted({p.stem for p in pathlib.Path("data/raw/images").iterdir()} - {p.stem for p in pathlib.Path("data/raw/labels").glob("*.txt")}))

In [ ]:
# 1d. Xem nhãn vẽ lên ảnh (~10 ảnh raw) -> ghi lại 3 lỗi tự nhìn thấy cho REPORT mục 4
from IPython.display import Image as IImage, display
IMGS = "img_0001 img_0004 img_0011 img_0020 img_0030 img_0032 img_0047 img_0083 img_0093 img_0112"
!python scripts/viz_labels.py {IMGS} --src data/raw
for s in IMGS.split():
    f = f"work/viz/{s}.png"
    if os.path.exists(f): print(s); display(IImage(f, width=640))

In [ ]:
# 1e. Vài dòng nhãn thô (bẫy: thiếu cột, nan, id 93/4/-1, box tràn)
import re
bad = []
for f in sorted(pathlib.Path("data/raw/labels").glob("*.txt")):
    for i, ln in enumerate(f.read_text(errors="replace").splitlines(), 1):
        p = ln.split()
        if ln.strip() and (len(p) != 5 or "nan" in ln or p[0] not in {"0","1","2","3"}):
            bad.append((f.stem, i, ln))
print(len(bad), "dòng nghi lỗi cú pháp, ví dụ:"); [print(b) for b in bad[:12]]

## Bước 2 — 4 module (TDD)
`label_format` → `gate` → `autofix` → `second_opinion`. Đích: **79 passed**.

In [ ]:
# 2a. Từng module
for t in ["label_format", "gate", "autofix", "second_opinion"]:
    print(f"== {t}")
    !python -m pytest tests/test_{t}.py -q 2>&1 | tail -1

In [ ]:
# 2b. Toàn bộ test
!python -m pytest -q 2>&1 | tail -3

## Bước 3 — Calibrate 2 ngưỡng
Đã chọn `duplicates.phash_dist = 12`, `second_opinion.conf = 0.6` (lý do: `submission/REPORT.md` mục 3).

In [ ]:
!grep -n "CALIBRATE" configs/pipeline.yaml
!python -m pipeline.calibrate --sweep duplicates.phash_dist=2,4,6,8,10,12,14,16,20 --sweep second_opinion.conf=0.1,0.25,0.4,0.5,0.6,0.7,0.8 --out submission/calibration

## Bước 4 — Chạy pipeline bằng MỘT lệnh
Mã thoát: 0 Release · 10 Hold · 20 Rework.

In [ ]:
rc = subprocess.run([sys.executable, "-m", "pipeline.run"]).returncode
print("\nMã thoát:", rc, {0: "Release", 10: "Hold", 20: "Rework"}.get(rc, "lỗi"))
!ls submission/reports

In [ ]:
# 4b. Đọc kết luận trước, chi tiết sau
import json, pandas as pd
s = json.load(open("submission/reports/08_summary.json")); print(json.dumps(s["verdict"], ensure_ascii=False, indent=2))
print("đã duyệt:", s["reviewed"], "/", s["budget"])
for f in ["02_error_list_v1", "04_error_list_v2", "07_error_list_v3"]:
    d = pd.read_csv(f"submission/reports/{f}.csv")
    print(f"\n== {f}"); print(d.groupby(["severity", "code"]).image.agg(["count", "nunique"]).rename(columns={"count": "dòng", "nunique": "ảnh"}))
print(pd.read_csv("submission/reports/03_fixes.csv").action.value_counts())

In [ ]:
# 4c. Chứng minh cửa chặn: không người duyệt -> Rework, mã 20, KHÔNG xuất
rc = subprocess.run([sys.executable, "-m", "pipeline.run", "--budget", "0", "--reports", "work/rep_nb", "--export-dir", "work/exp_nb"]).returncode
print("\nMã thoát:", rc, "| có thư mục export:", os.path.exists("work/exp_nb"))

## Bước 4 (tùy chọn, 15 điểm) — Train `raw` vs `clean` (yolo26n, 20 epoch, 2 seed, ~20 phút trên T4)

In [ ]:
# 4d. Dựng 2 bộ dữ liệu
import collections, pathlib
man = json.load(open("submission/export/clean_v1/manifest.json"))
print("quyết định:", man["decision"]["decision"], "| ảnh clean:", man["n_images"], "| ảnh raw:", len(list(pathlib.Path("data/raw/images").iterdir())))
for v in VARIANTS:
    subprocess.run([sys.executable, "scripts/materialize_dataset.py", "--variant", v, "--out", f"work/train/{v}", "--export", "submission/export/clean_v1"], check=True)
    cnt = collections.Counter(ln.split()[0] for f in pathlib.Path(f"work/train/{v}/train/labels").glob("*.txt") for ln in f.read_text().splitlines() if ln.split())
    print(v, dict(sorted(cnt.items())))

In [ ]:
# 4e. TRAIN
if DO_TRAIN and HAS_GPU:
    !python scripts/train_compare.py --variants {V_ARG} --seeds {S_ARG} --epochs {EPOCHS}
else:
    print("Bỏ qua train (không có GPU hoặc DO_TRAIN=False)")

## Bước 5 — Tổng hợp, kiểm bài nộp, tải kết quả

In [ ]:
# 5a. So sánh raw vs clean (TB ± độ lệch giữa seed)
from IPython.display import Markdown, Image, display
import glob
if glob.glob("submission/results/*_seed*.json"):
    !python scripts/summarize_results.py
    display(Markdown(open("submission/results/comparison.md").read())); display(Image("submission/results/comparison.png"))
    rows = [dict(variant=d["variant"], seed=d["seed"], train_images=d["train_images"], mAP50=d["map50"], **{f"AP50_{k}": v for k, v in d["ap50_per_class"].items()}) for d in (json.load(open(p)) for p in sorted(glob.glob("submission/results/*_seed*.json")))]
    display(pd.DataFrame(rows))
else:
    print("Chưa có kết quả train")

In [ ]:
# 5b. Kiểm bài nộp (đầy đủ nếu đã train, ngược lại check-core)
flag = "" if glob.glob("submission/results/*_seed*.json") else "--no-train"
!python scripts/check_submission.py {flag}

In [ ]:
# 5c. Tải kết quả train về máy -> giải nén vào submission/results/ rồi commit (KHÔNG sửa tay)
if glob.glob("submission/results/*_seed*.json"):
    !zip -qr results.zip submission/results
    from google.colab import files; files.download("results.zip")